# Universe — security master & exploratory analysis

**Step 2 of the KN Research Process.** This notebook answers three questions:

1. **What is in the investable universe?** `Investable_Universe.csv` carries only
   `ticker`, `name` and `isin` — the other seven columns ship empty. This notebook fills them
   from the data provider and profiles what comes back.
2. **What is wrong with the data we already downloaded?** The Data Curator has written
   `Data/Curator/Time_Series/`. section 4 reads those files back and builds a data-issues register,
   so the problems are known *before* a portfolio is built on top of them.
3. **What does the tradeable universe look like through time?** section 5 stacks the curated files
   into a panel and measures entries and exits — and, once the Curator computes an eligibility
   column and `SIGNAL_COLUMN` names it, the eligibility funnel, top-N set stability (the
   evidence behind a holding-count choice and an event-driven trigger frequency) and sector tilt.

The notebook is **self-contained** — its own imports and path resolution — so the universe can be
studied without running any other stage.

## Position in the pipeline

```
Universe (this notebook)  ->  Data/curator.py  ->  Experiments/  ->  Production/
    Security_Master.csv         Time_Series/        Portfolio, Backtest, Attribution
```

## ⚠ Sector and industry are a *current* snapshot, not point-in-time

The provider's profile endpoint returns the classification a security carries **today**. It has no
history. That matters because classifications move:

| When | What moved |
| --- | --- |
| Sep 2018 | GOOGL, GOOG, META, DIS, NFLX: Technology / Consumer Discretionary → Communication Services |
| Mar 2023 | Payment processors (V, MA, PYPL): Information Technology → Financials |

A 2015–2026 backtest bucketed on today's sector therefore **misattributes every year before the
reclassification**. Everything section 3 shows is "the universe as it is classified now" and is fine for
understanding composition — but sector *attribution* needs a **daily** sector series, which must
come out of `Data/Curator/` or `Data/Refinery/` as a per-date column. section 4 confirms that column does
not exist yet; section 6 records it as the blocking gap for the Attribution stage.

---

## 0 · Setup

Paths, provider selection, credentials. Nothing here touches the network.

In [ ]:
"""Stage 1 - Universe. Security master + exploratory analysis. Safe to run on its own."""
import collections
import concurrent.futures
import json
import os
import pathlib
import threading
import urllib.error
import urllib.parse
import urllib.request

import dotenv
import numpy
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory that holds pyproject.toml and Universe/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Universe").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
UNIVERSE_DIR = REPO_ROOT / "Universe"
CHART_DIR = UNIVERSE_DIR / "Charts"
PROVIDER_CACHE_DIR = UNIVERSE_DIR / "Provider_Cache"
TIME_SERIES_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"

RAW_UNIVERSE_PATH = UNIVERSE_DIR / "Investable_Universe.csv"
SECURITY_MASTER_PATH = UNIVERSE_DIR / "Security_Master.csv"
DATA_ISSUES_PATH = UNIVERSE_DIR / "Data_Issues.csv"

for directory in (CHART_DIR, PROVIDER_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# The provider the enrichment in section 2 runs against. Adding a provider means adding one fetch
# function and one normaliser in section 2.1 - see PROVIDER_ADAPTERS there.
DATA_PROVIDER = "financial_modeling_prep"
PROVIDER_TIMEOUT_SECONDS = 30

dotenv.load_dotenv(REPO_ROOT / "Config" / ".env")
PROVIDER_API_KEYS = {"financial_modeling_prep": os.getenv("KNDC_API_KEY_FMP")}

print(f"Repo root       : {REPO_ROOT}")
print(f"Raw universe    : {RAW_UNIVERSE_PATH.relative_to(REPO_ROOT)}")
print(f"Security master : {SECURITY_MASTER_PATH.relative_to(REPO_ROOT)}")
print(f"Curator data    : {TIME_SERIES_DIR.relative_to(REPO_ROOT)}"
      f" ({'present' if TIME_SERIES_DIR.is_dir() else 'MISSING - run Data/curator.py'})")
print(f"Provider        : {DATA_PROVIDER}"
      f" (key {'loaded' if PROVIDER_API_KEYS.get(DATA_PROVIDER) else 'MISSING'})")

---

## 1 · The raw investable universe

`Investable_Universe.csv` is the point-in-time ticker list — it deliberately keeps delisted,
renamed and acquired names so the backtest is not survivorship-biased.

Three things to establish before trusting it downstream:

1. **Which columns actually carry data.** Seven of the ten ship empty; section 2 fills what the provider
   can supply and records what it cannot.
2. **Shared ISINs** — the same company under two tickers. These are *ticker-change pairs*
   (`FISV`→`FI`, `SQ`→`XYZ`, `ANTM`→`ELV`), exactly what a point-in-time universe should contain,
   but the two legs must be **stitched into one position** or the backtest double-counts the name.
3. **Domicile** — the ISIN country prefix. Non-US domiciles carry different tax and settlement
   assumptions even though they all trade on US exchanges.

In [ ]:
# utf-8-sig strips the BOM the file starts with, otherwise the first column is named "\ufeffticker".
raw_universe = pandas.read_csv(RAW_UNIVERSE_PATH, encoding="utf-8-sig", dtype=str)

print(f"{RAW_UNIVERSE_PATH.name}: {raw_universe.shape[0]} rows x {raw_universe.shape[1]} columns\n")

# --- 1. Which columns carry data -----------------------------------------------------
population = pandas.DataFrame(
    {
        "non_empty": raw_universe.notna().sum(),
        "share": (raw_universe.notna().sum() / len(raw_universe)).map("{:.0%}".format),
        "example": [
            next((value for value in raw_universe[column] if pandas.notna(value)), "-")
            for column in raw_universe.columns
        ],
    }
)
print("Column population:")
print(population.to_string())

EMPTY_COLUMNS = tuple(population.index[population["non_empty"] == 0])
print(f"\n{len(EMPTY_COLUMNS)} empty column(s) for section 2 to fill: {', '.join(EMPTY_COLUMNS)}")

# --- 2. Identity: tickers and ISINs --------------------------------------------------
print(f"\nunique tickers : {raw_universe['ticker'].nunique()} of {len(raw_universe)} rows")
print(f"unique ISINs   : {raw_universe['isin'].nunique()}"
      f" (of {raw_universe['isin'].notna().sum()} non-null)")

duplicate_tickers = raw_universe[raw_universe["ticker"].duplicated(keep=False)]
print(f"duplicate ticker rows: {len(duplicate_tickers)}")

isin_pairs = (
    raw_universe[raw_universe["isin"].notna() & raw_universe["isin"].duplicated(keep=False)]
    .sort_values(["isin", "ticker"])
)
TICKER_CHANGE_GROUPS = (
    isin_pairs.groupby("isin")["ticker"].apply(lambda legs: " -> ".join(legs)).to_dict()
)
DISTINCT_COMPANIES = len(raw_universe) - len(isin_pairs) + len(TICKER_CHANGE_GROUPS)
print(
    f"\nShared-ISIN groups: {len(TICKER_CHANGE_GROUPS)} covering {len(isin_pairs)} rows"
    f" -> {DISTINCT_COMPANIES} distinct companies"
)
print(
    isin_pairs.groupby("isin")
    .agg(tickers=("ticker", lambda legs: ", ".join(sorted(legs))), company=("name", "first"))
    .to_string()
)

# --- 3. Domicile from the ISIN country prefix ----------------------------------------
DOMICILE_COUNTS = raw_universe["isin"].str[:2].fillna("(unknown)").value_counts()
print("\nDomicile (ISIN country prefix):")
print(DOMICILE_COUNTS.to_string())

UNIVERSE_TICKERS = tuple(raw_universe["ticker"].dropna().unique())
print(f"\nUNIVERSE_TICKERS ready ({len(UNIVERSE_TICKERS)} tickers)")

---

## 2 · Enrichment — build the security master

The raw file gives identity only. Everything else is fetched from the provider.

**Two-layer design.** The network layer caches the provider's *raw* payload to
`Universe/Provider_Cache/` as JSON Lines; the shaping layer derives `Security_Master.csv` from
that cache. Splitting them means re-running the notebook costs nothing, changing the column
mapping never triggers a refetch, and the untouched payload stays available for mining fields
this notebook does not yet use.

**Adding a provider** = one fetch function + one normaliser registered in `PROVIDER_ADAPTERS`.
No other cell changes.

### 2.1 · The provider adapter

`fetch` takes a ticker and returns the raw payload; `normalise` maps that payload onto the master
schema. `covers` is the honest list of master columns the provider can actually supply — the
columns *not* listed are a permanent gap for that provider, not a fetch failure.

In [ ]:
# --- Master schema ---------------------------------------------------------------------
# The ten raw columns (kept in their original order and meaning) plus what a provider can add.
MASTER_IDENTITY_COLUMNS = (
    "ticker", "name", "category", "active", "exchange",
    "currency", "isin", "figi", "cusips", "ric",
)
MASTER_PROVIDER_COLUMNS = (
    "sector", "industry", "country", "ipo_date", "market_cap",
    "beta", "average_volume", "cik", "employees", "exchange_full_name",
)
MASTER_COLUMNS = MASTER_IDENTITY_COLUMNS + MASTER_PROVIDER_COLUMNS

FINANCIAL_MODELING_PREP_PROFILE_URL = "https://financialmodelingprep.com/stable/profile"


def fetch_profile_financial_modeling_prep(ticker, api_key):
    """
    Return FMP's profile payload for `ticker`, or None when it has no profile.

    The endpoint takes one symbol per call - a comma-separated list returns an empty array
    rather than several records - so this is deliberately one request per ticker.
    """
    query = urllib.parse.urlencode({"symbol": ticker, "apikey": api_key})
    request = urllib.request.Request(f"{FINANCIAL_MODELING_PREP_PROFILE_URL}?{query}")
    with urllib.request.urlopen(request, timeout=PROVIDER_TIMEOUT_SECONDS) as response:
        payload = json.load(response)
    # Synthetic tickers (the KN600 benchmark) and symbols FMP does not carry return [].
    return payload[0] if payload else None


def normalise_profile_financial_modeling_prep(payload):
    """Map one FMP profile payload onto the master schema."""
    # FMP exposes ETF/fund/ADR as three booleans rather than one category string.
    category = "Stock"
    for flag, label in (("isEtf", "ETF"), ("isFund", "Fund"), ("isAdr", "ADR")):
        if payload.get(flag):
            category = label
    return {
        "name": payload.get("companyName"),
        "category": category,
        "active": payload.get("isActivelyTrading"),
        "exchange": payload.get("exchange"),
        "currency": payload.get("currency"),
        "isin": payload.get("isin"),
        "cusips": payload.get("cusip"),
        "sector": payload.get("sector"),
        "industry": payload.get("industry"),
        "country": payload.get("country"),
        "ipo_date": payload.get("ipoDate"),
        "market_cap": payload.get("marketCap"),
        "beta": payload.get("beta"),
        "average_volume": payload.get("averageVolume"),
        "cik": payload.get("cik"),
        "employees": payload.get("fullTimeEmployees"),
        "exchange_full_name": payload.get("exchangeFullName"),
    }


ProviderAdapter = collections.namedtuple("ProviderAdapter", ("fetch", "normalise", "covers"))

PROVIDER_ADAPTERS = {
    "financial_modeling_prep": ProviderAdapter(
        fetch=fetch_profile_financial_modeling_prep,
        normalise=normalise_profile_financial_modeling_prep,
        # `figi` and `ric` are absent from FMP's catalogue entirely. `ric` is a Refinitiv/LSEG
        # identifier, so the lseg_workspace provider is what would fill it.
        covers=(
            "name", "category", "active", "exchange", "currency", "isin", "cusips",
            *MASTER_PROVIDER_COLUMNS,
        ),
    ),
}

ADAPTER = PROVIDER_ADAPTERS[DATA_PROVIDER]
UNCOVERED_COLUMNS = tuple(column for column in MASTER_COLUMNS if column not in ADAPTER.covers)

print(f"Provider {DATA_PROVIDER!r} covers {len(ADAPTER.covers)}"
      f" of {len(MASTER_COLUMNS)} master columns.")
print(f"Cannot supply: {', '.join(UNCOVERED_COLUMNS)}")
print("  -> `ticker` comes from the raw file; `figi` / `ric` need a different provider.")

### 2.2 · Fetch (cached, resumable, threaded)

`RUN_ENRICHMENT = False` makes this cell purely offline: it reports what the cache already holds
and fetches nothing. Only tickers missing from the cache are ever requested, so an interrupted run
resumes where it stopped.

In [ ]:
# --- Switches --------------------------------------------------------------------------
RUN_ENRICHMENT = True        # False -> no network calls; report what the cache holds
REFETCH_ALL = False          # True  -> ignore the cache and refetch every ticker
PROVIDER_THREADS = 8         # concurrent requests; raise only if the plan's rate limit allows
PROGRESS_EVERY = 100

PROFILE_CACHE_PATH = PROVIDER_CACHE_DIR / f"{DATA_PROVIDER}_profiles.jsonl"

_cache_lock = threading.Lock()


def load_profile_cache(path):
    """Read the JSON Lines payload cache into {ticker: payload-or-None}."""
    if not path.is_file():
        return {}
    cached = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                cached[record["ticker"]] = record["payload"]
    return cached


def append_to_profile_cache(path, ticker, payload):
    """Append one payload to the cache. Appending (not rewriting) is what makes a run resumable."""
    with (
        _cache_lock,
        path.open("a", encoding="utf-8") as handle,
    ):
        handle.write(json.dumps({"ticker": ticker, "payload": payload}) + "\n")


def fetch_one_profile(ticker):
    """Fetch and cache `ticker`, returning (ticker, outcome). Never raises."""
    api_key = PROVIDER_API_KEYS[DATA_PROVIDER]
    try:
        payload = ADAPTER.fetch(ticker, api_key)
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError, OSError) as error:
        return ticker, f"failed: {type(error).__name__}: {error}"
    append_to_profile_cache(PROFILE_CACHE_PATH, ticker, payload)
    return ticker, "fetched" if payload else "no profile"


profile_cache = {} if REFETCH_ALL else load_profile_cache(PROFILE_CACHE_PATH)
pending = [ticker for ticker in UNIVERSE_TICKERS if ticker not in profile_cache]
print(f"Cache: {len(profile_cache)} payload(s) on disk, {len(pending)} ticker(s) pending.")

if RUN_ENRICHMENT and pending:
    if REFETCH_ALL and PROFILE_CACHE_PATH.is_file():
        PROFILE_CACHE_PATH.unlink()
    assert PROVIDER_API_KEYS.get(DATA_PROVIDER), f"no API key for {DATA_PROVIDER}; see Config/.env"

    outcomes = collections.Counter()
    print(f"Fetching {len(pending)} profile(s) on {PROVIDER_THREADS} thread(s)...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=PROVIDER_THREADS) as executor:
        futures = [executor.submit(fetch_one_profile, ticker) for ticker in pending]
        for done, future in enumerate(concurrent.futures.as_completed(futures), start=1):
            ticker, outcome = future.result()
            outcomes[outcome.split(":")[0]] += 1
            if outcome.startswith("failed"):
                print(f"  [FAILED] {ticker}: {outcome}")
            if done % PROGRESS_EVERY == 0 or done == len(pending):
                print(f"  [{done:>4}/{len(pending)}] {dict(outcomes)}")

    profile_cache = load_profile_cache(PROFILE_CACHE_PATH)
elif not RUN_ENRICHMENT:
    print("RUN_ENRICHMENT = False - no network calls.")

MISSING_PROFILES = tuple(ticker for ticker in UNIVERSE_TICKERS if not profile_cache.get(ticker))
print(f"\n{len(UNIVERSE_TICKERS) - len(MISSING_PROFILES)}/{len(UNIVERSE_TICKERS)}"
      " tickers have a profile.")
if MISSING_PROFILES:
    print(f"No profile ({len(MISSING_PROFILES)}): {', '.join(MISSING_PROFILES[:25])}"
          f"{' ...' if len(MISSING_PROFILES) > 25 else ''}")

### 2.3 · Shape the master and reconcile against the raw file

The raw file's `ticker` and `isin` are **authoritative** — they define the universe, so a provider
value never overwrites them. Instead the two are compared: an ISIN disagreement means the provider
has re-pointed a recycled ticker at a different company, which would silently corrupt any join
made on ticker alone.

In [ ]:
enriched_rows = []
for ticker in UNIVERSE_TICKERS:
    payload = profile_cache.get(ticker)
    row = dict.fromkeys(MASTER_COLUMNS)
    row["ticker"] = ticker
    if payload:
        row.update(ADAPTER.normalise(payload))
    enriched_rows.append(row)

security_master = pandas.DataFrame(enriched_rows, columns=MASTER_COLUMNS)

# --- Reconcile identity against the raw file -------------------------------------------
raw_indexed = raw_universe.set_index("ticker")
security_master["isin_provider"] = security_master["isin"]
security_master["isin"] = security_master["ticker"].map(raw_indexed["isin"])
security_master["name_raw"] = security_master["ticker"].map(raw_indexed["name"])

isin_conflicts = security_master[
    security_master["isin_provider"].notna()
    & (security_master["isin_provider"] != security_master["isin"])
]
print(f"ISIN disagreements (raw file vs provider): {len(isin_conflicts)}")
if len(isin_conflicts):
    print(
        isin_conflicts[["ticker", "name_raw", "isin", "name", "isin_provider"]]
        .to_string(index=False)
    )
    print("  -> the provider now points this ticker at a different security (recycled symbol).")
    print("     The raw file wins; never join on ticker alone across the whole history.")

# --- Types -----------------------------------------------------------------------------
security_master["active"] = security_master["active"].astype("boolean")
security_master["ipo_date"] = pandas.to_datetime(security_master["ipo_date"], errors="coerce")
for column in ("market_cap", "beta", "average_volume", "employees"):
    security_master[column] = pandas.to_numeric(security_master[column], errors="coerce")

security_master = security_master.drop(columns=["name_raw"])
security_master.to_csv(SECURITY_MASTER_PATH, index=False)

filled = security_master[list(MASTER_COLUMNS)].notna().sum()
print(f"\nSecurity master written: {SECURITY_MASTER_PATH.relative_to(REPO_ROOT)}"
      f" ({security_master.shape[0]} rows x {security_master.shape[1]} columns)")
print("\nColumn population after enrichment:")
print(
    pandas.DataFrame({
        "non_empty": filled,
        "share": (filled / len(security_master)).map("{:.0%}".format),
        "source": [
            "raw file" if column == "ticker"
            else "raw file (authoritative)" if column == "isin"
            else "unfilled - provider gap" if column in UNCOVERED_COLUMNS
            else DATA_PROVIDER
            for column in MASTER_COLUMNS
        ],
    }).to_string()
)
security_master.head(10)

---

## 3 · Composition — what the universe is made of

> **Read these as "the universe as classified today."** Per the header, sector and industry have
> no history in the provider, so these charts describe current composition, not the composition
> that was true in 2015.

Charts are single-series (one hue, values direct-labelled, no legend) and are saved as PNG to
`Universe/Charts/`.

### 3.1 · Chart helpers

In [ ]:
import matplotlib.pyplot

# --- Chart styling: one hue for magnitude, one accent for the highlighted mark ---------
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
SERIES_BLUE = "#2a78d6"
SERIES_ORANGE = "#eb6834"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": INK_SECONDARY,
    "text.color": INK_PRIMARY,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "font.size": 10,
    "axes.titlesize": 12,
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
})


def bar_counts(counts, title, subtitle=None, highlight=None, file_name=None, height_per_bar=0.34):
    """Horizontal count bars: largest on top, values direct-labelled, spines/grid recessive."""
    ordered = counts.sort_values()
    total = ordered.sum()
    colors = [
        SERIES_ORANGE if (highlight is not None and label in highlight) else SERIES_BLUE
        for label in ordered.index
    ]

    figure, axes = matplotlib.pyplot.subplots(figsize=(9, max(2.4, height_per_bar * len(ordered))))
    axes.barh(ordered.index.astype(str), ordered.to_numpy(), color=colors, height=0.72)

    span = ordered.max()
    for position, value in enumerate(ordered.to_numpy()):
        axes.text(
            value + span * 0.012, position, f"{value}  ({value / total:.1%})",
            va="center", ha="left", fontsize=9, color=INK_SECONDARY,
        )

    axes.set_xlim(0, span * 1.18)
    axes.set_xlabel("number of names")
    # Title sits above the subtitle line; the pad keeps the two from colliding.
    axes.set_title(title, loc="left", pad=24 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(
            subtitle, xy=(0, 1), xycoords="axes fraction",
            xytext=(0, 6), textcoords="offset points",
            fontsize=9, color=INK_SECONDARY, va="bottom", ha="left",
        )
    axes.xaxis.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right", "left"):
        axes.spines[side].set_visible(False)

    figure.tight_layout()
    if file_name:
        figure.savefig(CHART_DIR / file_name)
    matplotlib.pyplot.show()
    return ordered


def strip_axes(axes, sides=("top", "right")):
    """Drop the named spines - used everywhere a chart is not a bar_counts chart."""
    for side in sides:
        axes.spines[side].set_visible(False)


print("Chart helpers ready.")

### 3.2 · Listing status — the point-in-time payoff

`active` is what makes this universe usable for a backtest: the delisted and acquired names are
still in the file. If this split were 100% active, the universe would be survivorship-biased and
every backtest result built on it would be overstated.

In [ ]:
STATUS_COUNTS = (
    security_master["active"]
    .map({True: "actively trading", False: "delisted / acquired"})
    .fillna("(no profile)")
    .value_counts()
)

bar_counts(
    STATUS_COUNTS,
    "Listing status of the investable universe",
    subtitle=f"{len(security_master)} names - dead names retained,"
             " which is what removes survivorship bias",
    highlight={"delisted / acquired"},
    file_name="universe_listing_status.png",
    height_per_bar=0.55,
)

inactive_share = security_master["active"].eq(False).sum() / len(security_master)
print(f"Delisted / acquired: {inactive_share:.1%} of the universe.")
print("A survivorship-biased universe would show 0% here.")

### 3.3 · Sector composition

How the names split across sectors. This is the **equal-weight sector exposure** the strategy
starts from: whatever the eligibility rule does, it draws from this pool, so a sector that is 16% of
the universe can dominate the book if its names trend together.

In [ ]:
SECTOR_COUNTS = security_master["sector"].fillna("(unclassified)").value_counts()

bar_counts(
    SECTOR_COUNTS,
    "Universe composition by sector",
    subtitle=f"{len(security_master)} names - classification as of today, not point-in-time",
    highlight={SECTOR_COUNTS.idxmax()},
    file_name="universe_by_sector.png",
)

print(
    "Top-3 sector concentration: "
    f"{SECTOR_COUNTS.head(3).sum() / len(security_master):.1%} of the universe"
)

### 3.4 · Industry granularity

Two views: the **top 20 by count**, and a **concentration curve** answering "how many industries
do I need to cover 50% / 80% of the universe?". That number decides whether industry is usable as
an attribution bucket or has to be rolled up to sector.

In [ ]:
INDUSTRY_COUNTS = security_master["industry"].fillna("(unclassified)").value_counts()

bar_counts(
    INDUSTRY_COUNTS.head(20),
    "Top 20 industries by number of names",
    subtitle=f"{INDUSTRY_COUNTS.size} industries in total"
             f" - the other {max(INDUSTRY_COUNTS.size - 20, 0)}"
             f" hold {INDUSTRY_COUNTS.iloc[20:].sum()} names",
    file_name="universe_top20_industries.png",
)

# --- Concentration curve: cumulative share of names vs. number of industries -----------
cumulative_share = INDUSTRY_COUNTS.cumsum() / INDUSTRY_COUNTS.sum()
ranks = numpy.arange(1, len(cumulative_share) + 1)

figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.2))
axes.plot(ranks, cumulative_share.to_numpy(), color=SERIES_BLUE, linewidth=2)

for threshold in (0.5, 0.8):
    needed = int((cumulative_share < threshold).sum() + 1)
    axes.plot([needed], [cumulative_share.iloc[needed - 1]], "o", markersize=8,
              color=SERIES_ORANGE, markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)
    axes.annotate(
        f"{needed} industries = {threshold:.0%}",
        xy=(needed, cumulative_share.iloc[needed - 1]),
        xytext=(8, -14), textcoords="offset points",
        fontsize=9, color=INK_SECONDARY,
    )

axes.set_xlabel("industries, ranked by number of names")
axes.set_ylabel("cumulative share of universe")
axes.set_ylim(0, 1.02)
axes.set_xlim(0, len(cumulative_share))
axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes.set_title("How concentrated is the universe across industries?", loc="left", weight="bold")
axes.grid(True, color=GRID, linewidth=0.8)
axes.set_axisbelow(True)
strip_axes(axes)

figure.tight_layout()
figure.savefig(CHART_DIR / "universe_industry_concentration.png")
matplotlib.pyplot.show()

singletons = int((INDUSTRY_COUNTS == 1).sum())
print(f"{singletons} industries hold a single name; "
      f"median industry size = {INDUSTRY_COUNTS.median():.0f} names")

### 3.5 · Sector → industry structure (small multiples)

One panel per sector, top 6 industries each — small multiples rather than a stacked bar, because
11 sectors exceed any categorical palette and stacked segments would be unreadable. This shows
*how* each sector is built: a sector whose names sit in one industry (Utilities → Regulated
Electric) behaves like a single bet under a trend-following signal, while a broad sector
diversifies inside itself.

In [ ]:
TOP_INDUSTRIES_PER_SECTOR = 6

classified = security_master.dropna(subset=["sector", "industry"])
sectors_ordered = [
    sector for sector in SECTOR_COUNTS.index if sector in set(classified["sector"])
]

columns = 3
rows = -(-len(sectors_ordered) // columns)
figure, axes_grid = matplotlib.pyplot.subplots(rows, columns, figsize=(14, 2.9 * rows))
axes_flat = axes_grid.flatten()

for axes, sector in zip(axes_flat, sectors_ordered):
    counts = (
        classified.loc[classified["sector"] == sector, "industry"]
        .value_counts()
        .head(TOP_INDUSTRIES_PER_SECTOR)
        .sort_values()
    )
    labels = [label if len(label) <= 28 else label[:26] + "\u2026" for label in counts.index]
    axes.barh(labels, counts.to_numpy(), color=SERIES_BLUE, height=0.7)

    for position, value in enumerate(counts.to_numpy()):
        axes.text(value + counts.max() * 0.03, position, str(value),
                  va="center", fontsize=8, color=INK_SECONDARY)

    sector_total = int(SECTOR_COUNTS[sector])
    shown = int(counts.sum())
    axes.set_title(
        f"{sector}  \u00b7  {sector_total} names"
        + (f"  (top {len(counts)} = {shown / sector_total:.0%})" if shown < sector_total else ""),
        loc="left", fontsize=10, weight="bold",
    )
    axes.set_xlim(0, counts.max() * 1.25)
    axes.tick_params(axis="y", labelsize=8)
    axes.set_xticks([])
    strip_axes(axes, ("top", "right", "left", "bottom"))

for axes in axes_flat[len(sectors_ordered):]:
    axes.set_visible(False)

figure.suptitle(
    "Industry mix within each sector (top 6)", x=0.01, ha="left", fontsize=13, weight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.97))
figure.savefig(CHART_DIR / "universe_sector_industry_mix.png")
matplotlib.pyplot.show()

# Herfindahl of industry shares inside each sector: 1.0 = one industry, ->0 = well spread.
industry_hhi = (
    classified.groupby("sector")["industry"]
    .apply(lambda industries: ((industries.value_counts() / len(industries)) ** 2).sum())
    .sort_values(ascending=False)
    .round(3)
)
print("Industry concentration inside each sector (HHI, 1.0 = single industry):")
print(industry_hhi.to_string())

### 3.6 · Domicile, exchange and listing age

Domicile and exchange are settlement/tax facts rather than strategy inputs, but they decide
whether a name is actually tradeable in the account the strategy runs in. **Listing age** matters
more: a name that IPO'd in 2021 cannot produce a signal until its warm-up has passed, so
the effective universe in 2015 is much smaller than 786.

In [ ]:
bar_counts(
    DOMICILE_COUNTS,
    "Domicile by ISIN country prefix",
    subtitle="all names trade on US exchanges; non-US domiciles differ in tax/settlement treatment",
    highlight={code for code in DOMICILE_COUNTS.index if code != "US"},
    file_name="universe_by_domicile.png",
)

EXCHANGE_COUNTS = security_master["exchange"].fillna("(unknown)").value_counts()
bar_counts(
    EXCHANGE_COUNTS,
    "Listing exchange",
    subtitle="from the provider profile",
    file_name="universe_by_exchange.png",
    height_per_bar=0.45,
)

# --- Listing age: how much of the universe even exists at each point in the backtest ----
ipo_dates = security_master["ipo_date"].dropna().sort_values()
listed_count = pandas.Series(
    numpy.arange(1, len(ipo_dates) + 1), index=ipo_dates, name="listed",
)
backtest_window = listed_count[listed_count.index >= "2000-01-01"]

figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.2))
axes.plot(backtest_window.index, backtest_window.to_numpy(), color=SERIES_BLUE, linewidth=2)
axes.axvline(pandas.Timestamp("2015-01-01"), color=SERIES_ORANGE, linewidth=1.5, linestyle="--")
already_listed = int((ipo_dates < "2015-01-01").sum())
axes.annotate(
    f"backtest start: {already_listed} of {len(security_master)} names listed",
    xy=(pandas.Timestamp("2015-01-01"), already_listed),
    xytext=(10, -28), textcoords="offset points",
    fontsize=9, color=INK_SECONDARY,
)
axes.set_ylabel("names listed (cumulative)")
axes.set_title("How much of the universe exists over time?", loc="left", weight="bold")
axes.grid(True, color=GRID, linewidth=0.8)
axes.set_axisbelow(True)
strip_axes(axes)
figure.tight_layout()
figure.savefig(CHART_DIR / "universe_listing_age.png")
matplotlib.pyplot.show()

print(f"IPO date known for {len(ipo_dates)} of {len(security_master)} names.")
print(f"Listed before the 2015-01-01 backtest start: {already_listed}"
      f" ({already_listed / len(security_master):.1%})")

---

## 4 · Data-issues register — what the downloaded files actually contain

Everything above describes the universe *as catalogued*. This section reads the Curator's own
output back and asks whether the data behind each name is fit to trade on.

Each check writes a row into `Data_Issues.csv`, so Stage 2 has an explicit list to act on rather
than a chart to interpret.

### 4.1 · Coverage and per-file profile

One pass over `Data/Curator/Time_Series/`, reading only the columns the checks need. For every
ticker: does a file exist, what date range does it cover, how many rows, and — once a signal column exists —
how much of it is usable.

In [ ]:
# --- The strategy's columns: the only names in this notebook that are not process -----------
# SIGNAL_COLUMN is the 0/1 eligibility column the Curator computes for your strategy.  Leave it
# None until that column exists; sections 4.3 and 5.2-5.4 skip themselves and run once it is set.
SIGNAL_COLUMN = None  # e.g. "c_my_signal"
LIQUIDITY_COLUMN = "c_daily_traded_value_63d"
PRICE_COLUMN = "m_close_dividend_and_split_adjusted"
PROFILE_COLUMNS = [
    "m_date",
    PRICE_COLUMN,
    LIQUIDITY_COLUMN,
    *([SIGNAL_COLUMN] if SIGNAL_COLUMN else []),
]

BACKTEST_START = pandas.Timestamp("2015-01-01")


def read_header(path):
    """Column names of `path`, without reading a single data row."""
    return tuple(pandas.read_csv(path, nrows=0).columns)


MARKET_CALENDAR_TICKER = "SPY"


def load_market_calendar(directory, ticker):
    """
    The set of real trading sessions, taken from `ticker`'s own date column.

    A plain business-day range is the wrong yardstick - it counts every US market holiday as a
    missing session, which makes almost every name look gappy. An index ETF trades every session
    the market is open, so its dates *are* the calendar.
    """
    calendar_path = directory / f"{ticker}.csv"
    if not calendar_path.is_file():
        return None
    dates = pandas.read_csv(calendar_path, usecols=["m_date"], parse_dates=["m_date"])["m_date"]
    return pandas.DatetimeIndex(dates.sort_values().unique())


def profile_time_series(path, header, calendar):
    """
    Row-count, date span and null profile for one Curator output file.

    Only the columns `header` actually carries are requested: the directory is not guaranteed to
    hold one schema (see the schema check below), and a missing column must degrade to a null
    count rather than abort the whole pass.
    """
    present = [column for column in PROFILE_COLUMNS if column in header]
    frame = pandas.read_csv(path, usecols=present, parse_dates=["m_date"])
    dates = frame["m_date"]
    prices = frame[PRICE_COLUMN] if PRICE_COLUMN in frame else pandas.Series(dtype="float64")
    # Sessions the market was open between this name's own first and last observation, minus the
    # rows actually present: whatever is left is a hole in the middle of the series.
    if calendar is not None and len(dates):
        expected = calendar[(calendar >= dates.min()) & (calendar <= dates.max())]
        missing_sessions = max(len(expected) - dates.nunique(), 0)
    else:
        missing_sessions = 0
    return {
        "ticker": path.stem,
        "rows": len(frame),
        "columns": len(header),
        "first_date": dates.min(),
        "last_date": dates.max(),
        "missing_sessions": missing_sessions,
        "null_price": int(prices.isna().sum()) if len(prices) else len(frame),
        # No signal declared: nothing is unusable on signal grounds.  Declared but absent from
        # the file: every row is, which is the data-issues register's job to surface.
        "null_signal": (
            0 if SIGNAL_COLUMN is None
            else int(frame[SIGNAL_COLUMN].isna().sum()) if SIGNAL_COLUMN in frame
            else len(frame)
        ),
        "null_liquidity": (
            int(frame[LIQUIDITY_COLUMN].isna().sum()) if LIQUIDITY_COLUMN in frame else len(frame)
        ),
        "zero_or_negative_price": int((prices <= 0).sum()) if len(prices) else 0,
    }


time_series_paths = sorted(TIME_SERIES_DIR.glob("*.csv"))
print(f"Reading {len(time_series_paths)} file(s) from {TIME_SERIES_DIR.relative_to(REPO_ROOT)} ...")

# --- Schema drift ----------------------------------------------------------------------
# A directory holding two schemas is a silent trap: a column present in most files and absent
# in a few makes every downstream read conditional. Establish the majority schema first.
headers = {path.stem: read_header(path) for path in time_series_paths}
schema_groups = collections.Counter(headers.values())
MAJORITY_SCHEMA = schema_groups.most_common(1)[0][0]
OFF_SCHEMA = {
    ticker: tuple(sorted(set(MAJORITY_SCHEMA) - set(header)))
    for ticker, header in headers.items()
    if header != MAJORITY_SCHEMA
}

print()
print(f"Schemas in the directory: {len(schema_groups)}"
      f" (majority = {len(MAJORITY_SCHEMA)} columns, {schema_groups[MAJORITY_SCHEMA]} files)")
for ticker, missing in OFF_SCHEMA.items():
    print(f"  {ticker}: {len(headers[ticker])} columns,"
          f" missing {len(missing)} -> {', '.join(missing)}")

MARKET_CALENDAR = load_market_calendar(TIME_SERIES_DIR, MARKET_CALENDAR_TICKER)
print(f"Market calendar from {MARKET_CALENDAR_TICKER}: "
      + (f"{len(MARKET_CALENDAR)} sessions" if MARKET_CALENDAR is not None else "UNAVAILABLE"))

coverage = pandas.DataFrame(
    [profile_time_series(path, headers[path.stem], MARKET_CALENDAR) for path in time_series_paths]
)
coverage = coverage.set_index("ticker")

universe_set = set(UNIVERSE_TICKERS)
on_disk = set(coverage.index)
NON_UNIVERSE_FILES = tuple(sorted(on_disk - universe_set))
MISSING_FILES = tuple(sorted(universe_set - on_disk))

print(f"\nUniverse tickers with a data file : {len(universe_set & on_disk)}/{len(universe_set)}")
print(f"Files not in the universe         : {len(NON_UNIVERSE_FILES)}"
      f" ({', '.join(NON_UNIVERSE_FILES)})")
print("  -> benchmarks (SPY/QQQ/KN600) and the BIL cash proxy; expected, not an error.")
if MISSING_FILES:
    print(f"\nUniverse tickers with NO data file: {len(MISSING_FILES)}")
    print(f"  {', '.join(MISSING_FILES[:40])}{' ...' if len(MISSING_FILES) > 40 else ''}")

universe_coverage = coverage.loc[sorted(universe_set & on_disk)]
print(f"\nRow counts: min {universe_coverage['rows'].min()}, "
      f"median {universe_coverage['rows'].median():.0f}, max {universe_coverage['rows'].max()}")
print(f"Date span : {universe_coverage['first_date'].min().date()}"
      f" -> {universe_coverage['last_date'].max().date()}")
universe_coverage.head()

### 4.2 · History shape — short starts and truncated ends

Two failure modes look identical in a row count but mean opposite things:

- **A late first date** = the name had not listed yet (or the provider's history is short). It
  simply cannot be selected until the signal warms up.
- **An early last date** = the name stopped trading. The position must be *exited* on that date,
  and doing so uses information only available after the fact — the standard backtest compromise,
  but it needs to be visible rather than assumed.

In [ ]:
# The download itself starts at 2015-01-02, so comparing against BACKTEST_START would flag every
# name. The question that matters is which names start late *relative to the data as a whole*.
UNIVERSE_FIRST_DATE = universe_coverage["first_date"].min()
universe_last_date = universe_coverage["last_date"].max()
# Explicit unit: Timedelta("5D") and Timedelta(days=5) both hit a NumPy deprecation.
TOLERANCE = pandas.Timedelta(5, unit="D")

late_start = universe_coverage[universe_coverage["first_date"] > UNIVERSE_FIRST_DATE + TOLERANCE]
early_end = universe_coverage[universe_coverage["last_date"] < universe_last_date - TOLERANCE]

print(f"Data window: {UNIVERSE_FIRST_DATE.date()} -> {universe_last_date.date()}")
print(f"Names starting after {(UNIVERSE_FIRST_DATE + TOLERANCE).date()}: {len(late_start)}"
      f" ({len(late_start) / len(universe_coverage):.1%})"
      " - listed later, or the provider's history is short")
print(f"Names ending before {(universe_last_date - TOLERANCE).date()}: {len(early_end)}"
      f" ({len(early_end) / len(universe_coverage):.1%})"
      " - delisted, acquired, or a data gap")

print(f"Names present from the first session: {len(universe_coverage) - len(late_start)}")

# Cross-check the price data against the catalogue: a name the provider calls delisted should
# have a truncated series, and vice versa. Disagreements are where a backtest silently breaks.
status_by_ticker = security_master.set_index("ticker")["active"]
comparison = pandas.DataFrame({
    "series_ends_early": universe_coverage.index.isin(early_end.index),
    # `== False` rather than `~`: the map yields NaN for tickers the provider never
    # answered for, and only the explicit comparison reads those as "not delisted".
    "provider_says_delisted": universe_coverage.index.map(status_by_ticker) == False,  # noqa: E712
}, index=universe_coverage.index)

agreement = pandas.crosstab(
    comparison["series_ends_early"], comparison["provider_says_delisted"],
)
print("\nPrice history vs. provider listing status:")
print(agreement.to_string())

DISAGREEMENTS = comparison[comparison["series_ends_early"] != comparison["provider_says_delisted"]]
print(f"\nDisagreements: {len(DISAGREEMENTS)} name(s)")
print("  series ends early but provider says active -> data gap, not a delisting")
print("  provider says delisted but series runs to the end -> ticker reused by a live company")

figure, axes = matplotlib.pyplot.subplots(1, 2, figsize=(13, 4.0))
axes[0].hist(universe_coverage["first_date"], bins=40, color=SERIES_BLUE)
axes[0].axvline(BACKTEST_START, color=SERIES_ORANGE, linewidth=1.5, linestyle="--")
axes[0].set_title("First date in file", loc="left", weight="bold")
axes[1].hist(universe_coverage["last_date"], bins=40, color=SERIES_BLUE)
axes[1].set_title("Last date in file", loc="left", weight="bold")
for panel in axes:
    panel.set_ylabel("names")
    panel.grid(True, color=GRID, linewidth=0.8)
    panel.set_axisbelow(True)
    strip_axes(panel)
    panel.tick_params(axis="x", labelrotation=30)
figure.suptitle("History shape across the universe", x=0.01, ha="left", fontsize=13, weight="bold")
figure.tight_layout(rect=(0, 0, 1, 0.94))
figure.savefig(CHART_DIR / "universe_history_shape.png")
matplotlib.pyplot.show()

### 4.3 · Signal usability

A rolling signal needs its window before it produces anything, so every series opens with a
warm-up of nulls. That is expected. What is *not* expected is a name whose signal is null well
beyond the warm-up, or one so short it never produces a signal at all — those cannot be selected
and should be understood before they quietly shrink the tradeable universe. Set
`SIGNAL_WARMUP_ROWS` to the longest window behind your signal; the section runs once
`SIGNAL_COLUMN` is declared.

In [ ]:
if SIGNAL_COLUMN is None:
    never_usable = universe_coverage.iloc[0:0]
    beyond_warmup = universe_coverage.iloc[0:0]
    print("Skipped: no SIGNAL_COLUMN declared in section 4.1.")
    print("Nothing is unusable on signal grounds.")
else:
    SIGNAL_WARMUP_ROWS = 0  # e.g. 200 for a 200-day moving average

    usable = universe_coverage.assign(
        usable_signal_rows=universe_coverage["rows"] - universe_coverage["null_signal"],
        excess_nulls=universe_coverage["null_signal"] - SIGNAL_WARMUP_ROWS + 1,
    )

    never_usable = usable[usable["usable_signal_rows"] <= 0]
    beyond_warmup = usable[usable["excess_nulls"] > 0]

    print(f"Names that never produce a signal      : {len(never_usable)}")
    if len(never_usable):
        print(f"  {', '.join(never_usable.index[:30])}{' ...' if len(never_usable) > 30 else ''}")
    print(f"Names with nulls beyond the warm-up   : {len(beyond_warmup)}")
    if len(beyond_warmup):
        print(
            beyond_warmup.sort_values("excess_nulls", ascending=False)
            .head(15)[["rows", "null_signal", "excess_nulls", "first_date", "last_date"]]
            .to_string()
        )

    print(f"\nNames with null prices        : {int((universe_coverage['null_price'] > 0).sum())}")
    non_positive = int((universe_coverage["zero_or_negative_price"] > 0).sum())
    print(f"Names with non-positive prices: {non_positive}")
    gappy = universe_coverage[universe_coverage["missing_sessions"] > 0]
    print(f"Names with internal date gaps : {len(gappy)}"
          f" (against {MARKET_CALENDAR_TICKER}'s real trading calendar)")
    if len(gappy):
        print(
            gappy.sort_values("missing_sessions", ascending=False)
            .head(15)[["rows", "missing_sessions", "first_date", "last_date"]]
            .to_string()
        )


### 4.4 · The classification gap

The check that decides whether Stage 5 can be trusted: **do the Curator files carry a sector or
industry column?** If they do not, the only classification available is section 2's current snapshot, and
every sector attribution before a reclassification date is wrong.

In [ ]:
sample_header = pandas.read_csv(time_series_paths[0], nrows=0).columns
CLASSIFICATION_COLUMNS = tuple(
    column for column in sample_header
    if any(token in column.lower() for token in ("sector", "industry", "classification"))
)

print(f"Curator output columns: {len(sample_header)}")
print(f"Classification columns in the time series: {len(CLASSIFICATION_COLUMNS)}")
if CLASSIFICATION_COLUMNS:
    print(f"  {', '.join(CLASSIFICATION_COLUMNS)}")
else:
    print("  NONE. Historical sector/industry is unavailable anywhere in the repo.")
    print()
    print("  Consequence: sector attribution can only use section 2's *current* snapshot, which")
    print("  misattributes every period before a reclassification. Known movers inside the")
    print("  backtest window:")
    print("    2018-09  GOOGL, GOOG, META, DIS, NFLX -> Communication Services")
    print("    2023-03  V, MA, PYPL and peers        -> Financials")
    print()
    print("  Fix belongs in Stage 2: a daily sector/industry column carried per date in")
    print("  Data/Curator/ or derived in Data/Refinery/.")

# Names whose current sector is known to differ from their historical one, so the size of the
# problem is concrete rather than theoretical.
KNOWN_RECLASSIFIED = ("GOOGL", "GOOG", "META", "DIS", "NFLX", "V", "MA", "PYPL")
affected = security_master[security_master["ticker"].isin(KNOWN_RECLASSIFIED)]
if len(affected):
    print("\nNames in this universe known to have been reclassified (current sector shown):")
    print(affected[["ticker", "name", "sector", "industry"]].to_string(index=False))

### 4.5 · Write the register

One row per issue, with the ticker list attached, so Stage 2 can work through it.

In [ ]:
def issue_row(check, severity, tickers, note):
    """One row of the register. `tickers` is truncated in the CSV but counted in full."""
    tickers = tuple(tickers)
    return {
        "check": check,
        "severity": severity,
        "count": len(tickers),
        "note": note,
        "tickers": ", ".join(tickers[:50]) + (" ..." if len(tickers) > 50 else ""),
    }


data_issues = pandas.DataFrame([
    issue_row(
        "no_data_file", "blocking", MISSING_FILES,
        "in the universe but never downloaded - rerun Data/curator.py for these",
    ),
    issue_row(
        "no_provider_profile", "warning", MISSING_PROFILES,
        "no profile from the provider - no sector/industry, so unbucketable in attribution",
    ),
    issue_row(
        "no_historical_classification", "blocking",
        () if CLASSIFICATION_COLUMNS else tuple(KNOWN_RECLASSIFIED),
        "Curator files carry no daily sector column; attribution before a reclassification"
        " is wrong. Listed tickers are known movers, not the full set",
    ),
    issue_row(
        "never_produces_signal", "warning", never_usable.index,
        "history shorter than the signal's warm-up - can never be selected",
    ),
    issue_row(
        "nulls_beyond_warmup", "warning", beyond_warmup.index,
        "signal nulls exceed the warm-up - gaps in the underlying price series",
    ),
    issue_row(
        "series_ends_early", "info", early_end.index,
        "price history stops before the universe's last date - delisting or a data gap",
    ),
    issue_row(
        "status_disagreement", "warning", DISAGREEMENTS.index,
        "price history and provider listing status disagree - recycled ticker or data gap",
    ),
    issue_row(
        "isin_conflict", "warning", isin_conflicts["ticker"],
        "provider ISIN differs from the universe file - the ticker now points at another security",
    ),
    issue_row(
        "internal_date_gaps", "info",
        gappy.sort_values("missing_sessions", ascending=False).index,
        "sessions missing inside the name's own date range, measured against"
        f" {MARKET_CALENDAR_TICKER}'s trading calendar",
    ),
    issue_row(
        "schema_drift", "warning",
        tuple(ticker for ticker in OFF_SCHEMA if ticker in universe_set),
        "file does not carry the majority column set - any read of a missing column breaks."
        " Off-schema files outside the universe (the KN600 benchmark) are excluded here",
    ),
    issue_row(
        "non_positive_price", "blocking",
        universe_coverage.index[universe_coverage["zero_or_negative_price"] > 0],
        "zero or negative adjusted close - breaks return calculation",
    ),
])

SEVERITY_RANK = {"blocking": 0, "warning": 1, "info": 2}
data_issues = (
    data_issues.assign(severity_rank=data_issues["severity"].map(SEVERITY_RANK))
    .sort_values(["severity_rank", "count"], ascending=[True, False])
    .drop(columns=["severity_rank"])
)
data_issues.to_csv(DATA_ISSUES_PATH, index=False)

print(f"Data-issues register written: {DATA_ISSUES_PATH.relative_to(REPO_ROOT)}\n")
print(data_issues[["check", "severity", "count", "note"]].to_string(index=False))

---

## 5 · The tradeable universe through time

Everything above is either a catalogue snapshot (section 3) or a per-file audit (section 4). This section stacks
the Curator files into one compact panel and asks the questions the portfolio stage actually
depends on:

1. **Entries and exits** — how many names carry data as time passes, and when do names appear and
   disappear? This is the survivorship-bias picture, quantified.
2. **The eligibility funnel** — of the names with data on a date, how many have a measurable
   signal, and how many are eligible?
3. **The top-N liquidity set** — how stable is the group of names a top-N-by-liquidity rule
   would actually hold, and how often does its composition change? Under an event-driven
   rebalancing rule (see `BLUEPRINT_1.md`) this set's stability *is* the strategy's turnover.
4. **Sector tilt of the top-N slice** — a liquidity-ranked book is not sector-neutral; this
   shows what it is structurally tilted toward before any weighting is applied.

Sections 5.2 to 5.4 read `SIGNAL_COLUMN` and skip themselves until it is declared in section 4.1.

In [ ]:
# A compact panel: one row per name per date, only the columns these sections need.
# Roughly 2M rows x 4 columns - loads in under a minute and fits comfortably in memory.
PANEL_COLUMNS = ["m_date", LIQUIDITY_COLUMN, *([SIGNAL_COLUMN] if SIGNAL_COLUMN else [])]

universe_frames = []
for path in time_series_paths:
    if path.stem not in universe_set:
        continue
    present = [column for column in PANEL_COLUMNS if column in headers[path.stem]]
    frame = pandas.read_csv(path, usecols=present, parse_dates=["m_date"])
    frame.insert(0, "ticker", path.stem)
    universe_frames.append(frame)

universe_panel = (
    pandas.concat(universe_frames, ignore_index=True)
    .sort_values(["m_date", "ticker"])
    .reset_index(drop=True)
)

print(f"Panel: {universe_panel['ticker'].nunique()} tickers x "
      f"{universe_panel['m_date'].nunique()} dates = {len(universe_panel):,} rows")

### 5.1 · Entries and exits — the point-in-time payoff, quantified

`first_date` and `last_date` per name (from section 4.1) turned into a timeline: how many names appear
each year (IPO, or the provider's history begins) and how many disappear (delisting or
acquisition), with the count of names carrying data through time. A survivorship-biased universe
would show entries only — the exits are what make this backtest honest.

In [ ]:
first_dates = universe_coverage["first_date"]
last_dates = universe_coverage["last_date"]
window_first_date = first_dates.min()
window_last_date = last_dates.max()

# Names present from the first session are openers, not entries; names alive at the end are
# survivors, not exits. The tolerance mirrors section 4.2.
entries_by_year = (
    first_dates[first_dates > window_first_date + TOLERANCE].dt.year.value_counts().sort_index()
)
exits_by_year = (
    last_dates[last_dates < window_last_date - TOLERANCE].dt.year.value_counts().sort_index()
)
names_with_data = universe_panel.groupby("m_date")["ticker"].size()

flow_years = sorted(set(entries_by_year.index) | set(exits_by_year.index))
figure, (axes_flow, axes_level) = matplotlib.pyplot.subplots(
    2, 1, figsize=(9, 6.6), gridspec_kw={"height_ratios": [1, 1]},
)
axes_flow.bar(
    [year - 0.2 for year in flow_years],
    entries_by_year.reindex(flow_years, fill_value=0).to_numpy(),
    width=0.4, color=SERIES_BLUE, label="entries",
)
axes_flow.bar(
    [year + 0.2 for year in flow_years],
    exits_by_year.reindex(flow_years, fill_value=0).to_numpy(),
    width=0.4, color=SERIES_ORANGE, label="exits",
)
axes_flow.set_title("Names entering and leaving the data, per year", loc="left", weight="bold")
axes_flow.set_ylabel("names")
axes_flow.legend(frameon=False, loc="upper left")
axes_flow.grid(True, axis="y", color=GRID, linewidth=0.8)
axes_flow.set_axisbelow(True)
strip_axes(axes_flow)

axes_level.plot(names_with_data.index, names_with_data.to_numpy(), color=SERIES_BLUE, linewidth=2)
axes_level.set_title("Names carrying a data row, through time", loc="left", weight="bold")
axes_level.set_ylabel("names")
axes_level.grid(True, color=GRID, linewidth=0.8)
axes_level.set_axisbelow(True)
strip_axes(axes_level)

figure.tight_layout()
figure.savefig(CHART_DIR / "universe_entries_exits.png")
matplotlib.pyplot.show()

print(f"Openers (present from {window_first_date.date()}): "
      f"{int((first_dates <= window_first_date + TOLERANCE).sum())}")
print(f"Entries during the window : {int(entries_by_year.sum())}")
print(f"Exits during the window   : {int(exits_by_year.sum())}"
      f" ({int(exits_by_year.sum()) / len(universe_coverage):.1%} of names with data)")

### 5.2 · The eligibility funnel

On each date: names with a data row → names whose signal is measurable (past its warm-up) →
names actually eligible. The gap between the first two lines is the structural cost of the
warm-up; the gap between the last two is the market regime (the analyzer's breadth reading is
exactly the ratio of the two). The bottom line is the pool a top-N selection draws from — if it
ever approaches N, selection stops being selective.

In [ ]:
if SIGNAL_COLUMN is None:
    print("Skipped: no SIGNAL_COLUMN declared in section 4.1.")
else:
    # --- Strategy parameters for 5.2-5.4: the holding counts a top-N rule would compare --------
    HOLDING_COUNTS = (20, 30)
    CHOSEN_COUNT = 30

    funnel = universe_panel.groupby("m_date").agg(
        with_data=("ticker", "size"),
        measurable=(SIGNAL_COLUMN, "count"),
        eligible=(SIGNAL_COLUMN, "sum"),
    )

    figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.6))
    axes.plot(funnel.index, funnel["with_data"].to_numpy(),
              color=INK_SECONDARY, linewidth=1.4, label="with data")
    axes.plot(funnel.index, funnel["measurable"].to_numpy(),
              color=SERIES_BLUE, linewidth=1.8, label="signal measurable")
    axes.plot(funnel.index, funnel["eligible"].to_numpy(),
              color=SERIES_ORANGE, linewidth=1.8, label="eligible")
    axes.axhline(CHOSEN_COUNT, color=INK_SECONDARY, linewidth=1.0, linestyle=":")
    axes.annotate(f"top-{CHOSEN_COUNT} selection depth",
                  xy=(funnel.index[len(funnel) // 20], CHOSEN_COUNT),
                  xytext=(0, 5), textcoords="offset points", fontsize=8, color=INK_SECONDARY)
    axes.set_ylabel("names")
    axes.set_title("Eligibility funnel: data -> measurable -> eligible", loc="left", weight="bold")
    axes.legend(frameon=False, loc="upper left")
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    strip_axes(axes)
    figure.tight_layout()
    figure.savefig(CHART_DIR / "universe_eligibility_funnel.png")
    matplotlib.pyplot.show()

    WARMUP_CUTOFF = funnel.index.min() + pandas.Timedelta(300, unit="D")
    post_warmup_funnel = funnel[funnel.index >= WARMUP_CUTOFF]
    print(f"Post warm-up ({WARMUP_CUTOFF.date()} onward):")
    print(f"  eligible pool: min {int(post_warmup_funnel['eligible'].min())},"
          f" median {post_warmup_funnel['eligible'].median():.0f},"
          f" max {int(post_warmup_funnel['eligible'].max())}")
    thinnest_date = post_warmup_funnel["eligible"].idxmin()
    print(f"  thinnest pool on {thinnest_date.date()}"
          f" ({int(post_warmup_funnel.loc[thinnest_date, 'eligible'])} eligible names)")


### 5.3 · The top-N liquidity set — stability and trigger frequency

If the rule holds the top N names by 63-day ADTV among the eligible ones, and rebalances only
when that set changes, then the set's stability decides the strategy's turnover, and the
comparison between the counts in `HOLDING_COUNTS` is the evidence behind the top-N choice. Measured here: how often the set changes, how many names are replaced when it does,
and how much of today's set survives a week / month / quarter / year.

In [ ]:
if SIGNAL_COLUMN is None:
    print("Skipped: no SIGNAL_COLUMN declared in section 4.1.")
else:
    eligible = universe_panel[
        (universe_panel[SIGNAL_COLUMN] == 1.0)
        & universe_panel[LIQUIDITY_COLUMN].notna()
    ].copy()
    # method="first" breaks ties deterministically, so the sets are identical on every run.
    eligible["liquidity_rank"] = (
        eligible.groupby("m_date")[LIQUIDITY_COLUMN]
        .rank(ascending=False, method="first")
    )

    SURVIVAL_HORIZONS = {"1 week": 5, "1 month": 21, "1 quarter": 63, "1 year": 252}

    stability_rows = []
    daily_additions_by_count = {}
    for holding_count in HOLDING_COUNTS:
        top_sets = (
            eligible[eligible["liquidity_rank"] <= holding_count]
            .groupby("m_date")["ticker"]
            .agg(frozenset)
            .sort_index()
        )
        additions = pandas.Series(
            [
                len(current_set - previous_set)
                for current_set, previous_set in zip(top_sets.iloc[1:], top_sets.iloc[:-1])
            ],
            index=top_sets.index[1:],
        )
        daily_additions_by_count[holding_count] = additions

        survival = {}
        for label, horizon in SURVIVAL_HORIZONS.items():
            overlaps = [
                len(future_set & today_set) / len(today_set)
                for future_set, today_set in zip(top_sets.iloc[horizon:], top_sets.iloc[:-horizon])
                if len(today_set) > 0
            ]
            survival[label] = float(numpy.mean(overlaps))

        trigger_days = additions[additions > 0]
        stability_rows.append({
            "top_N": holding_count,
            "avg_set_size": top_sets.map(len).mean(),
            "trigger_day_share": (additions > 0).mean(),
            "avg_names_replaced_on_trigger": trigger_days.mean() if len(trigger_days) else 0.0,
            **{f"survives_{label}": share for label, share in survival.items()},
        })

    stability = pandas.DataFrame(stability_rows).set_index("top_N")
    print("Top-N set stability (event-driven rebalancing evidence):")
    print(stability.round(3).to_string())

    # Rolling view of how busy the trigger rule is, for the N the strategy will run.
    rolling_additions = daily_additions_by_count[CHOSEN_COUNT].rolling(63).mean()
    figure, axes = matplotlib.pyplot.subplots(figsize=(9, 4.2))
    axes.plot(rolling_additions.index, rolling_additions.to_numpy(),
              color=SERIES_BLUE, linewidth=1.8)
    axes.set_ylabel("names replaced per day (63d avg)")
    axes.set_title(
        f"How busy is the trigger rule? Top-{CHOSEN_COUNT} replacements, smoothed",
        loc="left", weight="bold",
    )
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    strip_axes(axes)
    figure.tight_layout()
    figure.savefig(CHART_DIR / "universe_topN_turnover.png")
    matplotlib.pyplot.show()

    print(f"\nReading: a trigger-day share of "
          f"{stability.loc[CHOSEN_COUNT, 'trigger_day_share']:.0%} means the book trades roughly "
          f"{stability.loc[CHOSEN_COUNT, 'trigger_day_share'] * 252:.0f} days a year;"
          " whether that is cheap or expensive is settled net-of-cost in the backtest.")


### 5.4 · Sector tilt of the top-N slice

Name-days spent in the top-N set per sector, against each sector's share of the whole universe.
The difference is the structural tilt a liquidity-ranked book carries before weighting even starts.
Classification is today's snapshot — the header warning applies — so this is a diagnostic, not an
attribution.

In [ ]:
if SIGNAL_COLUMN is None:
    print("Skipped: no SIGNAL_COLUMN declared in section 4.1.")
else:
    top_members = eligible[eligible["liquidity_rank"] <= CHOSEN_COUNT]
    sector_by_ticker = security_master.set_index("ticker")["sector"]

    top_slice_share = (
        top_members["ticker"]
        .map(sector_by_ticker)
        .fillna("(unclassified)")
        .value_counts(normalize=True)
    )
    universe_share = SECTOR_COUNTS / SECTOR_COUNTS.sum()

    sector_tilt = (
        pandas.DataFrame({
            "topN_name_days": top_slice_share,
            "universe_names": universe_share,
        })
        .fillna(0.0)
    )
    sector_tilt["tilt"] = sector_tilt["topN_name_days"] - sector_tilt["universe_names"]
    sector_tilt = sector_tilt.sort_values("tilt")

    figure, axes = matplotlib.pyplot.subplots(figsize=(9, max(2.8, 0.38 * len(sector_tilt))))
    tilt_colors = [
        SERIES_BLUE if value >= 0 else SERIES_ORANGE
        for value in sector_tilt["tilt"].to_numpy()
    ]
    axes.barh(sector_tilt.index.astype(str), sector_tilt["tilt"].to_numpy(),
              color=tilt_colors, height=0.72)
    axes.axvline(0, color=INK_SECONDARY, linewidth=1.0)
    axes.set_xlabel(f"top-{CHOSEN_COUNT} name-day share minus universe share")
    axes.xaxis.set_major_formatter(lambda value, _: f"{value:+.0%}")
    axes.set_title(f"What the top-{CHOSEN_COUNT} liquidity slice is tilted toward",
                   loc="left", weight="bold")
    axes.xaxis.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right", "left"):
        axes.spines[side].set_visible(False)
    figure.tight_layout()
    figure.savefig(CHART_DIR / "universe_topN_sector_tilt.png")
    matplotlib.pyplot.show()

    print(sector_tilt.map("{:.1%}".format).to_string())


---

## 6 · Handoff

What this notebook produces and what the next stage consumes.

| Output | Consumed by |
| --- | --- |
| `Universe/Security_Master.csv` | `Data/curator.py` (identifier list), Experiments (sector buckets) |
| `Universe/Data_Issues.csv` | Stage 2 — the work list |
| `Universe/Charts/*.png` | `FINDINGS_1.md` |
| Section 5.3 stability table | Experiment 1 — evidence for the holding count and trigger frequency |
| `Universe/Provider_Cache/*.jsonl` | this notebook only (regenerable) |

`Security_Master.csv`, `Data_Issues.csv` and `Provider_Cache/` are all **regenerable** and
therefore gitignored; `Investable_Universe.csv` is the only committed input.

In [ ]:
tradeable = security_master[
    security_master["ticker"].isin(universe_coverage.index)
    & ~security_master["ticker"].isin(never_usable.index)
]
CURATOR_IDENTIFIERS = tuple(tradeable["ticker"])

summary = pandas.DataFrame({
    "metric": [
        "rows in raw universe",
        "distinct companies (by ISIN)",
        "with a provider profile",
        "sectors",
        "industries",
        "actively trading",
        "delisted / acquired",
        "with a Curator data file",
        "producing a usable signal" if SIGNAL_COLUMN else "with data (no signal declared yet)",
        "blocking data issues",
    ],
    "value": [
        len(raw_universe),
        DISTINCT_COMPANIES,
        len(UNIVERSE_TICKERS) - len(MISSING_PROFILES),
        int(security_master["sector"].nunique()),
        int(security_master["industry"].nunique()),
        int(security_master["active"].eq(True).sum()),
        int(security_master["active"].eq(False).sum()),
        len(universe_coverage),
        len(CURATOR_IDENTIFIERS),
        int(data_issues.loc[data_issues["severity"] == "blocking", "count"].sum()),
    ],
})
print(summary.to_string(index=False))

print(f"\nCharts written to        : {CHART_DIR.relative_to(REPO_ROOT)}")
print(f"CURATOR_IDENTIFIERS ready: {len(CURATOR_IDENTIFIERS)} tickers")
print(f"  first 10: {', '.join(CURATOR_IDENTIFIERS[:10])}")

---

## Open items for Stage 2 (Data)

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **Daily sector/industry column.** Neither the provider profile nor the Curator files carry classification history. | Blocks trustworthy sector attribution for the whole 2015–2026 window. |
| 2 | **`figi` and `ric` stay empty.** FMP does not carry either; `ric` needs the LSEG provider. | Cross-provider joins have to go through ISIN until then. |
| 3 | **Work the blocking rows of `Data_Issues.csv`.** | They are the names a backtest would silently mishandle. |
| 4 | **Per-ticker exit building blocks** (distance from the 252-day high, realized vol) belong in `Data/Refinery/custom_calculations.py` via its documented per-ticker exception — not in the Curator, where a schema change forces a full refetch. | Experiment 2's take-profit exits need them. |
